# 🤖 EP2 — Agente Ferrandes (Ferretería Industrial Andes)

**ISY0101 · Ingeniería de Soluciones con IA · Evaluación Parcial N°2**

Este notebook ejecuta el repositorio completo en Google Colab:

1. Copia el repositorio (GitHub) o sube el ZIP de **Ferrandes_Agente_EP2**.
2. Ejecuta las **38 pruebas unitarias** (sin instalar nada: solo librería estándar).
3. Ejecuta la **demo** que genera la evidencia (`evidencia/`).
4. (Opcional) Smoke test del agente **LangChain** con `pip install -r requirements.txt`.
5. **Guarda los resultados** en Google Drive o los descarga a tu PC.

> ⚠️ En Colab la carpeta `/content` es temporal: al cerrar la sesión se borra,
> por eso el paso 5 (guardar/descargar) es obligatorio antes de salir.


In [17]:
# 1) Copiar el repositorio
# Opción A — CLONAR DESDE GITHUB (reemplaza la URL por la de tu repo):

URL_REPO = "https://github.com/jghleo/Ferrandes_Agente_EP2"   # ← cambia esto

!git clone "$URL_REPO" ferrandes 2>&1 | tail -2
%cd ferrandes
!ls

Cloning into 'ferrandes'...
/content/ferrandes/ferrandes
assets	evidencia  presentacion  requirements.txt  src
colab	informe    README.md	 scripts	   tests


In [18]:
# 1b) Opción B — SUBIR EL ZIP (si no usaste Git)
# Arrastra "Proyecto_EP2_Ferrandes.zip" al panel de archivos (izquierda) y corre esta celda:

import os, zipfile, glob

zips = sorted(glob.glob("/content/*.zip"))
print("ZIPs detectados:", zips)
if os.path.exists("/content/ferrandes/README.md"):
    print("Ya hay un repo clonado en /content/ferrandes: no hace falta descomprimir.")
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/content")
    carpeta = "/content/Ferrandes_Agente_EP2"
    %cd {carpeta}
    print("Descomprimido. Contenido:", len(os.listdir(".")), "elementos")
else:
    print("No se encontró ZIP. Arrastra el archivo o clona el repo en el paso 1a.")

%pwd

ZIPs detectados: ['/content/EP2_Ferrandes_resultados.zip']
Ya hay un repo clonado en /content/ferrandes: no hace falta descomprimir.


'/content/ferrandes/ferrandes'

## 2) Pruebas unitarias del agente

Ejecuta las 38 pruebas: herramientas, memoria de corto/largo plazo,
recuperación semántica (TF-IDF y métricas), planificación, decisiones
adaptativas y replanificación.


In [19]:
# 2) Pruebas unitarias
!python tests/ejecutar_tests.py


test_consulta_flujo_pedido_completo (test_agente.TestAgente.test_consulta_flujo_pedido_completo) ... ok
test_consulta_simple_stock (test_agente.TestAgente.test_consulta_simple_stock) ... ok
test_derivacion_normativa (test_agente.TestAgente.test_derivacion_normativa) ... ok
test_interaccion_registrada_en_largo_plazo (test_agente.TestAgente.test_interaccion_registrada_en_largo_plazo) ... ok
test_memoria_corto_plazo_activa (test_agente.TestAgente.test_memoria_corto_plazo_activa) ... ok
test_buscar_catalogo (test_herramientas.TestConsulta.test_buscar_catalogo) ... ok
test_consultar_precio_con_iva (test_herramientas.TestConsulta.test_consultar_precio_con_iva) ... ok
test_consultar_stock (test_herramientas.TestConsulta.test_consultar_stock) ... ok
test_consultar_stock_bajo_avisa_reposicion (test_herramientas.TestConsulta.test_consultar_stock_bajo_avisa_reposicion) ... ok
test_ficha_tecnica (test_herramientas.TestConsulta.test_ficha_tecnica) ... ok
test_crear_cotizacion_con_descuento (test_he

## 3) Demo + evidencia

`scripts/demo.py` ejecuta 11 escenarios de negocio (stock, pedidos con
reposición automática, crédito, cliente nuevo, guardrail normativo, memoria,
RAG y replanificación ante ítems inválidos) y genera:

- `evidencia/evidencia_pruebas.txt` — salida completa
- `evidencia/logs_agente.jsonl` — trazabilidad por consulta
- `datos_ferrandes.db` — memoria de largo plazo (SQLite)


In [20]:
# 3) Demo completa (genera la evidencia)
!python scripts/demo.py > /tmp/demo_salida.txt 2>&1
# mostramos un resumen de la salida
!tail -25 /tmp/demo_salida.txt
print("\nArchivos de evidencia:")
!ls -la evidencia/

  DECISION REPLANIFICACION: Items descartados por invalidos: FIJ-999 (producto desconocido), ELE-001 (cantidad <= 0). -- El plan se reajusta y continua solo con los items validos.
  DECISION EVALUACION_CREDITO: CREDITO_OK. Constructora Montemar Ltda. opera con credito; cupo disponible $3,800,000 cubre el total $714. -- Politica de credito y cupo aplicada de forma adaptativa segun el cliente.
  DECISION APRUEBA: Pedido confirmado y programado para despacho. -- Cupo disponible y condiciones de la operacion cumplidas.
  Cotizacion: COT-000004 | Total: $714
  Pedido: PAT-000002

ESCENARIO H2 - ENTRADA AMBIGUA (producto/cliente inexistente)
CONSULTA: tienes stock del producto TOR-999?
------------------------------------------------------------------------------
No existe el producto TOR-999 en el catalogo.

RESUMEN DE TRAZABILIDAD

RESUMEN DE TRAZABILIDAD
- Interacciones registradas en memoria de largo plazo: 10
- Cotizaciones generadas: 4 | Pedidos: 2 | Tickets de escalamiento: 3 | Ordene

## 4) (Opcional) Agente LangChain

Solo si quieres mostrar la orquestación con el **framework LangChain**
(`create_agent`) o usar Claude con tu API key. El resto del proyecto funciona
sin estos paquetes.


In [21]:
# 4) Instalar LangChain (opcional, ~1 min) y correr smoke test
!pip install -q -r requirements.txt
!python scripts/smoke_langchain.py


SMOKE TEST AGENTE LANGCHAIN (create_agent) - FERRANDES EP2
Modo: OFFLINE (modelo simulado) | herramientas: StructuredTool
GRAFO DE ORQUESTACION (intents):
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | model |           
          +-------+.          
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 

CONSULTA: tienes stock del taladro HER-001?
[human] 'tienes stock del taladro HER-001?'
[ai] tooll_call -> consultar_stock({'producto_id': 'HER-001'})
[tool] name=consultar_stock respuesta='Stock de HER-001 - Taladro percutor 13 mm, 850 W (categoria: Herramientas, unidad: unidad): 34 unidades (stock minimo recomendado: 10). Estado: OK.'
[ai] name=AgenteFerrandes respuesta='Res

In [22]:
# 4b) (Opcional) Otra consulta sobre el agente local (chat rápido)
import sys; sys.path.insert(0, ".")
from src.agente import AgenteFerrandes
agente = AgenteFerrandes()
print(agente.responder("tienes stock del taladro HER-001?"))
print()
print(agente.responder("cotizar CL-001 con TOR-001:100 y EPP-001:50"))
print()
print(agente.responder("que EPP me exige la normativa para trabajo en altura?"))


Stock de HER-001 - Taladro percutor 13 mm, 850 W (categoria: Herramientas, unidad: unidad): 34 unidades (stock minimo recomendado: 10). Estado: OK.

FLUJO DE PEDIDO (cliente CL-001) -> PEDIDO_CONFIRMADO
  [1. ANALIZAR_SOLICITUD] (analizar_solicitud): 2 items validos para Constructora Montemar Ltda..
  [2. VALIDAR_STOCK] (consultar_stock): TOR-001: stock 850/200
  [2. VALIDAR_STOCK] (consultar_stock): EPP-001: stock 400/100
  [4. CREAR_COTIZACION] (crear_cotizacion): Cotizacion COT-000003 generada para Constructora Montemar Ltda. (CL-001):
- 100 x TOR-001 Tornillo hexagonal inoxidable A4-70 M8 x 30 mm = $12,000
- 50 x EPP-001 Casco de seguridad clase E color amarillo = $449,500
Subtotal: $461,500 | Descuento: 0% ($0) | IVA 19%: $87,685 | TOTAL: $549,185.
  [5. EVALUAR_RIESGO_CREDITO] (evaluar_riesgo_credito): DECISION: CREDITO_OK. Constructora Montemar Ltda. opera con credito; cupo disponible $3,800,000 cubre el total $549,185.
  [6. DECIDIR_FLUJO_FINAL] (confirmar_pedido): Pedido PAT-0

## 4c) (Opcional) Embeddings densos y métricas del retriever (IE4)

Compara el retriever TF-IDF (por defecto) con embeddings reales sobre el mismo
conjunto etiquetado. En Colab la descarga del modelo es rápida.


In [23]:
# 4c) Embeddings densos + comparación de métricas
!pip install -q sentence-transformers
import sys; sys.path.insert(0, ".")
from src.datos import cargar_datos
from src.buscar_contexto import evaluar_recuperacion

datos = cargar_datos()
print("TF-IDF    :", evaluar_recuperacion(datos, modo="tfidf"))
print("Embeddings:", evaluar_recuperacion(datos, modo="embeddings"))
# Para que TODO el agente use embeddings, activa la variable de entorno:
# import os; os.environ["FERRANDES_RETRIEVER_MODE"] = "auto"


TF-IDF    : {'n_casos': 8, 'top_k': 3, 'modo': 'tfidf', 'hit_rate': 1.0, 'mrr': 1.0, 'detalle': [{'consulta': 'stock de tornillo inoxidable', 'esperado': 'TOR-001', 'obtenido': ['TOR-001', 'TOR-002'], 'hit': True}, {'consulta': 'cuantas horas demora el despacho en concepcion', 'esperado': 'faq_despacho', 'obtenido': ['faq_despacho', 'pol_despacho', 'pol_ubicacion'], 'hit': True}, {'consulta': 'horario de atencion', 'esperado': 'pol_horario', 'obtenido': ['pol_horario'], 'hit': True}, {'consulta': 'como devuelvo un producto', 'esperado': 'faq_devolucion', 'obtenido': ['faq_devolucion', 'pol_devoluciones', 'faq_ficha'], 'hit': True}, {'consulta': 'donde estan ubicados', 'esperado': 'pol_ubicacion', 'obtenido': ['pol_ubicacion'], 'hit': True}, {'consulta': 'que EPP entregan', 'esperado': 'faq_epp', 'obtenido': ['faq_epp', 'pol_devoluciones', 'EPP-001'], 'hit': True}, {'consulta': 'necesito un taladro percutor', 'esperado': 'HER-001', 'obtenido': ['HER-001'], 'hit': True}, {'consulta': 'pu

## 5) GUARDAR los resultados 🔒

**Antes de cerrar la sesión** descarga los artefactos a tu PC o guárdalos en
Google Drive. En Colab la carpeta `/content` se borra al terminar.


In [24]:
# 5a) Descargar artefactos a tu PC (se descarga cada archivo)
from google.colab import files

import os
archivos = [
    "evidencia/evidencia_pruebas.txt",
    "evidencia/logs_agente.jsonl",
    "informe/Informe_Tecnico_EP2_Ferrandes.docx",
    "presentacion/Presentacion_EP2_Ferrandes.pptx",
]
for a in archivos:
    if os.path.exists(a):
        files.download(a)
        print("Descargado:", a)
    else:
        print("No existe (se genera en pasos 3/4):", a)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargado: evidencia/evidencia_pruebas.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargado: evidencia/logs_agente.jsonl


... (2 outputs left out)

Descargado: informe/Informe_Tecnico_EP2_Ferrandes.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargado: presentacion/Presentacion_EP2_Ferrandes.pptx


In [25]:
# 5b) Guardar TODO en Google Drive (con copia completa de la carpeta evidencia)
from google.colab import drive
drive.mount("/content/drive")   # acepta la autorización en el navegador

import shutil, os, glob
dst = "/content/drive/MyDrive/EP2_Ferrandes/resultados"
os.makedirs(dst, exist_ok=True)

for carpeta in ["evidencia", "assets", "informe", "presentacion"]:
    if os.path.isdir(carpeta):
        shutil.copytree(carpeta, f"{dst}/{carpeta}", dirs_exist_ok=True)

if os.path.exists("datos_ferrandes.db"):
    shutil.copy("datos_ferrandes.db", f"{dst}/datos_ferrandes.db")

print("Guardado en Google Drive:")
!ls -la "$dst"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Guardado en Google Drive:
total 76
drwx------ 6 root root  4096 Oct 21 21:22 .
drwx------ 3 root root  4096 Oct 21 18:55 ..
drwx------ 2 root root  4096 Oct 21 21:21 assets
-rw------- 1 root root 53248 Oct 21 21:22 datos_ferrandes.db
drwx------ 2 root root  4096 Oct 21 21:21 evidencia
drwx------ 2 root root  4096 Oct 21 21:21 informe
drwx------ 2 root root  4096 Oct 21 21:21 presentacion


In [26]:
# 5c) Crear un ZIP con todo y descargarlo (método más simple)
import shutil, os
shutil.make_archive("/content/EP2_Ferrandes_resultados", "zip",
                    root_dir=".", base_dir=".")
from google.colab import files
files.download("/content/EP2_Ferrandes_resultados.zip")
print("ZIP generado y descargado ✅")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ZIP generado y descargado ✅
